In [ ]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.optimizers import Adam

In [ ]:
# Paths
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

os.makedirs(MODELS_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 7
EPOCHS = 20

In [ ]:
# Load split CSVs
train_df = pd.read_csv(os.path.join(RESULTS_DIR, "train_split.csv"))
val_df = pd.read_csv(os.path.join(RESULTS_DIR, "val_split.csv"))

In [ ]:
# Compute class weights
classes = np.sort(train_df["label"].unique())
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_df["label"]
)
class_weights = {k: min(v, 4.0) for k, v in class_weights.items()}

print("Class weights:")
for k, v in class_weights.items():
    print(f"Class {k}: {v:.4f}")


In [ ]:
# Image loader
def load_and_preprocess_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, IMG_SIZE)
    image = image / 255.0
    return image, label

In [ ]:
# Build datasets
train_ds = tf.data.Dataset.from_tensor_slices(
    (train_df["path"].values, train_df["label"].values)
)
val_ds = tf.data.Dataset.from_tensor_slices(
    (val_df["path"].values, val_df["label"].values)
)

train_ds = train_ds.map(load_and_preprocess_image).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.map(load_and_preprocess_image).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [ ]:
# Data augmentation
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.03),
])

In [ ]:
# Base model
base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

In [ ]:
# Unfreeze top 10 layers
base_model.trainable = True
for layer in base_model.layers[:-10]:
    layer.trainable = False


In [ ]:
# Build full model
inputs = tf.keras.Input(shape=(224, 224, 3))

In [ ]:
# Apply augmentation
x = data_augmentation(inputs)

In [ ]:
# Pass through ResNet
x = base_model(x, training=False)

In [ ]:
# HEAD (your improved part)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(MODELS_DIR, "best_model.keras"),
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks
)

model.save(os.path.join(MODELS_DIR, "resnet50_finetuned.keras"))
print("Fine-tuned model saved.")